# Selected 6 Planets — Longest Continuous Run per Sector
**UHJ:** WASP-078, WASP-072, TOI-4773  
**HJ:** TrES-5, CoRoT-01, HATS-13

For each planet and each available sector, the **longest single continuous run** (no gap > 5 min) is highlighted in full color. All other points in the sector are shown in light gray.

In [ ]:
import warnings
import numpy as np
import matplotlib.pyplot as plt
import lightkurve as lk

warnings.filterwarnings('ignore')

MAX_GAP_MIN = 5.0

PLANETS = {
    'WASP-078': dict(kind='UHJ', color='#c0392b', P=2.175, depth=0.811),
    'WASP-072': dict(kind='UHJ', color='#c0392b', P=2.217, depth=0.434),
    'TOI-4773': dict(kind='UHJ', color='#c0392b', P=1.745, depth=0.811),
    'TrEs-5':   dict(kind='HJ',  color='#1a6b3c', P=1.482, depth=1.998),
    'CoRoT-01': dict(kind='HJ',  color='#1a6b3c', P=1.509, depth=1.986),
    'HATS-13':  dict(kind='HJ',  color='#1a6b3c', P=3.044, depth=1.972),
}

def clean_lc(lc, sigma=3.0):
    lc   = lc.remove_nans().remove_outliers(sigma=sigma)
    time = np.asarray(lc.time.value, dtype=float)
    flux = np.asarray(lc.flux.value, dtype=float)
    flux = flux / np.nanmedian(flux)
    return time, flux

def find_segments(time, max_gap_min=5.0):
    gap_d  = max_gap_min / 1440.0
    dt     = np.diff(time)
    breaks = np.where(dt > gap_d)[0]
    starts = np.concatenate([[0], breaks + 1])
    ends   = np.concatenate([breaks, [len(time) - 1]])
    return list(zip(starts.tolist(), ends.tolist()))

print('Setup complete.')

In [ ]:
for planet, cfg in PLANETS.items():
    P          = cfg['P']
    depth      = cfg['depth']
    color      = cfg['color']
    depth_frac = depth / 100.0

    print(f'[{planet}] Downloading...', flush=True)
    try:
        sr = lk.search_lightcurve(planet, mission='TESS', cadence=120, author='SPOC')
        if len(sr) == 0:
            print('  No SPOC 2-min data — skipped.'); continue
        lc_collection = sr.download_all(quality_bitmask='default')
    except Exception as e:
        print(f'  Download error: {e}'); continue

    print(f'  {len(lc_collection)} sector(s) found.', flush=True)

    for lc_raw in lc_collection:
        sector = lc_raw.meta.get('SECTOR', '?')
        try:
            time, flux = clean_lc(lc_raw)
        except Exception as e:
            print(f'  S{sector} error: {e}'); continue

        segs    = find_segments(time, MAX_GAP_MIN)
        lengths = [e - s + 1 for s, e in segs]
        best_i  = int(np.argmax(lengths))
        bs, be  = segs[best_i]
        best_duration_h = (time[be] - time[bs]) * 24

        span_d  = time[-1] - time[0]
        n_trans = round(span_d / P)

        fig, ax = plt.subplots(figsize=(18, 4))

        # 1. All points — light gray background
        ax.plot(time, flux, '.', ms=1.2, color='#bbbbbb', alpha=0.5, zorder=1)

        # 2. Longest run — full color, bold
        ax.plot(time[bs:be+1], flux[bs:be+1],
                '.', ms=2.0, color=color, alpha=0.85, zorder=3,
                label=f'Longest run: {lengths[best_i]} pts | {best_duration_h:.1f} h')

        # 3. Shorter runs — colored but faded so they're distinguishable from gray
        for i, (s, e) in enumerate(segs):
            if i == best_i:
                continue
            ax.plot(time[s:e+1], flux[s:e+1],
                    '.', ms=1.2, color=color, alpha=0.2, zorder=2)

        # Y limits
        flux_p1  = np.nanpercentile(flux, 1)
        flux_p99 = np.nanpercentile(flux, 99)
        margin   = max(3 * depth_frac, (flux_p99 - flux_p1) * 0.1, 0.002)
        ax.set_ylim(min(flux_p1, 1.0 - 2*depth_frac) - margin*0.3,
                    max(flux_p99, 1.0 + depth_frac)   + margin*0.3)

        ax.set_xlabel('BTJD', fontsize=11)
        ax.set_ylabel('Norm. Flux', fontsize=11)
        ax.set_title(
            f'{planet} ({cfg["kind"]})  |  Sector {sector}  |  '
            f'{span_d:.1f} d  |  {len(segs)} run(s)  |  ~{n_trans} transits',
            fontsize=12, fontweight='bold'
        )
        ax.yaxis.set_major_formatter(plt.FormatStrFormatter('%.4f'))
        ax.tick_params(labelsize=9)
        ax.legend(fontsize=9, loc='upper right')

        plt.tight_layout()
        plt.show()

        # Print run breakdown
        run_info = '  Runs: ' + ' | '.join(
            f'{"[LONGEST] " if i==best_i else ""}run{i+1}: {e-s+1} pts ({(time[e]-time[s])*24:.1f} h)'
            for i, (s, e) in enumerate(segs)
        )
        print(f'  S{sector}: {run_info}', flush=True)

    print(f'  Done.\n', flush=True)